# 16-caching-persistence

16-caching-persistence/01-caching-and-persistence.py

Demonstrates cache(), persist(), StorageLevels, catalog caching,
and the cost/benefit tradeoffs of caching in PySpark.

Run:
    python 01-caching-and-persistence.py
Spark UI: http://localhost:4040  →  Storage tab shows cached RDDs/DataFrames

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# ── SparkSession ──────────────────────────────────────────────────────────────
spark = get_spark("16-caching-persistence")
dfs   = register_views(spark)
emp   = dfs["employees"]
sal   = dfs["salary_history"]


# ── Section 1: cache() vs persist() ──────────────────────────────────────────
# cache() is shorthand for persist(StorageLevel.MEMORY_AND_DISK)
# persist() accepts an explicit StorageLevel for fine-grained control
print("\n" + "="*55)
print("  Section 1: cache() vs persist()")
print("="*55)

from pyspark.storagelevel import StorageLevel
import time

emp_cached   = emp.cache()                                    # MEMORY_AND_DISK (default)
emp_persisted = sal.persist(StorageLevel.MEMORY_AND_DISK_2)  # replicated to 2 nodes

print("emp_cached    storage level:", emp_cached.storageLevel)
print("emp_persisted storage level:", emp_persisted.storageLevel)


# ── Section 2: Cache materialization ─────────────────────────────────────────
# First action writes data to cache; subsequent actions read from cache.
# In local mode with tiny data the gap is small; on GB datasets it is huge.
print("\n" + "="*55)
print("  Section 2: Cache materialization — cold vs warm read")
print("="*55)

t1 = time.time(); emp_cached.count(); t_cold = time.time() - t1
t2 = time.time(); emp_cached.count(); t_warm = time.time() - t2
print(f"Cold read (cache not yet filled): {t_cold:.3f}s")
print(f"Warm read (served from cache):    {t_warm:.3f}s")
# Spark UI → Storage tab: 'employees' appears after the first action above


# ── Section 3: is_cached and Spark UI verification ───────────────────────────
print("\n" + "="*55)
print("  Section 3: is_cached flag + Spark UI Storage tab")
print("="*55)

print("emp_cached.is_cached:", emp_cached.is_cached)
emp_cached.count()  # already materialized; confirms is_cached stays True
# Check http://localhost:4040/storage — you should see the DataFrame listed
# with fraction cached, memory used, and storage level.


# ── Section 4: StorageLevels compared ────────────────────────────────────────
# MEMORY_ONLY       — deserialized in JVM heap; fastest; OOM risk if data > RAM
# MEMORY_AND_DISK   — spills to disk if RAM is insufficient (safe default)
# MEMORY_AND_DISK_2 — same + replicated across 2 executors for fault tolerance
# DISK_ONLY         — always on disk; no OOM; slower than memory
# OFF_HEAP          — requires spark.memory.offHeap.enabled=true; avoids GC
print("\n" + "="*55)
print("  Section 4: StorageLevel comparison")
print("="*55)

levels = [
    ("MEMORY_ONLY",     StorageLevel.MEMORY_ONLY),
    ("MEMORY_AND_DISK", StorageLevel.MEMORY_AND_DISK),
    ("DISK_ONLY",       StorageLevel.DISK_ONLY),
]
for name, level in levels:
    df_p = emp.persist(level)
    df_p.count()   # materialize so the Storage tab reflects actual usage
    print(f"StorageLevel: {name:20s} | useDisk={level.useDisk} | useMemory={level.useMemory}")
    df_p.unpersist()


# ── Section 5: unpersist() ────────────────────────────────────────────────────
# unpersist() removes the cached data immediately (blocking by default).
# After unpersist, the DataFrame is recomputed on next action.
print("\n" + "="*55)
print("  Section 5: unpersist()")
print("="*55)

emp_cached.unpersist()
print("After unpersist, emp_cached.is_cached:", emp_cached.is_cached)
# Spark UI → Storage tab: 'employees' entry disappears


# ── Section 6: When to cache — cost/benefit ───────────────────────────────────
# CACHE when the SAME DataFrame is consumed by multiple downstream actions.
# Caching a DataFrame that is only used once wastes memory for no gain.
# Here: active_emp is used in 3 actions → caching pays off.
print("\n" + "="*55)
print("  Section 6: When to cache (multi-action reuse)")
print("="*55)

active_emp = emp.filter(F.col("status") == "Active").cache()

active_emp.count()                                                          # action 1 — materializes cache
active_emp.groupBy("dept_id").agg(F.avg("salary").alias("avg_salary")).show()  # action 2 — reads from cache
active_emp.orderBy("salary").select("emp_id", "first_name", "salary").show(5)  # action 3 — reads from cache

active_emp.unpersist()
# Spark UI → Jobs tab: actions 2 and 3 show 0 bytes read from storage (cache hit)


# ── Section 7: Caching a joined DataFrame ────────────────────────────────────
# Cache the result of an expensive join so downstream aggregations
# do not re-execute the join each time.
print("\n" + "="*55)
print("  Section 7: Caching a joined DataFrame")
print("="*55)

active_with_dept = (
    emp.filter(F.col("status") == "Active")
       .join(dfs["departments"], "dept_id", "left")
       .select("emp_id", "first_name", "dept_name", "salary")
       .cache()
)
active_with_dept.count()  # materialize — join executes once
active_with_dept.groupBy("dept_name").agg(
    F.count("*").alias("head_count"),
    F.avg("salary").alias("avg_salary")
).show()
# Spark UI → SQL tab: second query shows FromCache node, no ShuffleExchange
active_with_dept.unpersist()


# ── Section 8: spark.catalog.cacheTable and uncacheTable ─────────────────────
# cacheTable caches the Spark SQL temp view for use with spark.sql().
# Useful when mixing DataFrame API and SQL in the same script.
print("\n" + "="*55)
print("  Section 8: catalog.cacheTable / uncacheTable")
print("="*55)

spark.catalog.cacheTable("employees")
spark.sql("SELECT dept_id, COUNT(*) AS cnt FROM employees GROUP BY dept_id ORDER BY dept_id").show()
# Spark UI → Storage tab: table 'employees' listed under Cached Tables
spark.catalog.uncacheTable("employees")
print("employees view uncached")


# ── Section 9: clearCache (drop all cached data) ─────────────────────────────
# spark.catalog.clearCache() removes ALL cached tables and DataFrames at once.
# Use in notebooks to reclaim memory before a new pipeline run.
print("\n" + "="*55)
print("  Section 9: clearCache()")
print("="*55)

emp.cache().count()
sal.cache().count()
print("Cached objects before clearCache — check http://localhost:4040/storage")
spark.catalog.clearCache()
print("After clearCache — all cached DataFrames released")
print("emp.is_cached:", emp.is_cached, "| sal.is_cached:", sal.is_cached)


# ── Done ──────────────────────────────────────────────────────────────────────
stop_and_wait(spark)